# Model Compare

**Compare two models head-to-head on your own prompt, with your own keys.** Responses, token cost, and latency, side by side. 60 seconds, free, no signup.

A free, own-key tool by [pysolvr / LLM Prover](https://llmprover.com).

---

> **Free tool, provided as-is, no warranty.** It uses *your own* API keys and calls the providers directly. **Your keys never leave your machine / your Colab account and are never sent to pysolvr.** You are responsible for your own usage and costs. Outputs are informational, not advice. pysolvr is not liable for decisions made from these results.

**Key safety:** this notebook never asks you to type a key into a cell (see step 2) — keys live in Colab Secrets or your shell environment, so a saved or shared copy of this notebook contains no secrets. If a key ever does end up in a file anywhere, rotate it at the provider.

## 1. Setup

Pulls the canonical tool from the repo and installs the provider SDKs. `openai` also covers Gemini, Grok, Together, and Groq via their OpenAI-compatible endpoints.

In [ ]:
!pip -q install openai>=1.0 anthropic>=0.30
!wget -q -O compare_core.py https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/compare/compare_core.py
!wget -q -O prices.json      https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/compare/prices.json
print('ready')

## 2. Add your API keys (do NOT type them into a cell)

**Never paste an API key directly into a notebook cell.** A key typed into a cell gets saved into the notebook file, your Colab revision history, and any copy you share or commit. That is the most common way keys leak. Store them *outside* the notebook instead. The cell below only *reads* them, so your keys never live in the notebook.

### On Colab (recommended) — use the Secrets panel
1. Click the **key icon** (🔑 Secrets) in the left sidebar.
2. **Add a new secret** for each provider you need. Use these exact names:

   | Secret name | Models it unlocks |
   |-------------|-------------------|
   | `OPENAI_API_KEY` | `gpt-*`, `o3`/`o4-*` |
   | `ANTHROPIC_API_KEY` | `claude*` |
   | `GEMINI_API_KEY` | `gemini*` |
   | `XAI_API_KEY` | `grok*` |
   | `TOGETHER_API_KEY` | `<org>/<model>` |
   | `GROQ_API_KEY` | `qwen/*` |

3. Toggle **Notebook access** on for each secret you added. You only need the two providers whose models you are comparing.

Colab stores these encrypted, per-user, tied to your account — never written into the notebook. That is why this is safe to share: the keys are not in the file.

### On local Jupyter — use environment variables
Set them in your shell *before* launching Jupyter (not in a cell):
```bash
export OPENAI_API_KEY=sk-...
export ANTHROPIC_API_KEY=sk-ant-...
jupyter lab
```

Then run the cell below — it loads whatever you configured (Colab Secrets first, then environment variables). **Nothing is sent to pysolvr; keys stay on your machine / in your Colab account.**

In [ ]:
# Loads keys from Colab Secrets (preferred) or the environment. It does NOT contain any key
# itself -- add them via the Secrets panel / your shell as described above, then run this.
import os

_KEY_NAMES = [
    'OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY',
    'XAI_API_KEY', 'TOGETHER_API_KEY', 'GROQ_API_KEY',
]

# On Colab, read from the encrypted Secrets store and expose to the provider SDKs via env.
try:
    from google.colab import userdata  # present only on Colab
    for _name in _KEY_NAMES:
        try:
            _val = userdata.get(_name)
            if _val:
                os.environ[_name] = _val
        except Exception:
            pass  # secret not set or notebook access off -- fine, skip it
except ImportError:
    pass  # not on Colab -- keys come from the shell environment (local Jupyter)

_loaded = [n for n in _KEY_NAMES if os.environ.get(n)]
if _loaded:
    print('keys loaded for:', _loaded)
else:
    print('No keys found. Add them via the Colab Secrets panel (key icon, left) with Notebook')
    print('access ON, or set them as environment variables before launching Jupyter. See cell above.')

## 3. Pick two models + your prompt

Use the provider's own model-ID string (the tool is model-agnostic — it compares whatever your key reaches, so it never goes stale). Example IDs by provider:

| Provider (key) | Example model IDs |
|----------------|-------------------|
| OpenAI (`OPENAI_API_KEY`) | `gpt-5`, `gpt-5-mini`, `gpt-4o`, `o4-mini` |
| Anthropic (`ANTHROPIC_API_KEY`) | `claude-opus-5-5`, `claude-sonnet-5-5`, `claude-haiku-5-5` |
| Google (`GEMINI_API_KEY`) | `gemini-4-argon`, `gemini-3.8-flash` |
| xAI (`XAI_API_KEY`) | `grok-4.7`, `grok-4.6` |
| Together (`TOGETHER_API_KEY`) | `deepseek-ai/DeepSeek-V4-Pro-0813`, `moonshotai/Kimi-K3` |
| Groq (`GROQ_API_KEY`) | `qwen/qwen3.8-27b` |

These are examples, not a fixed list — providers add and retire models constantly. **To find what exists right now and what it costs:**
- Current models + capabilities across providers: **[llmprover.com/models](https://llmprover.com/models/)**
- Live per-token pricing (per provider): **[llmprover.com/api-pricing](https://llmprover.com/api-pricing/)**

Set `MODEL_A` and `MODEL_B` to the two you want to compare, and write your prompt.

*Launched from a model-pair page? `MODEL_A` / `MODEL_B` are pre-filled for you (the `?models=a,b` convention) — just set your keys and prompt and run.*

In [ ]:
MODEL_A = 'gpt-5'
MODEL_B = 'claude-opus-5-5'
PROMPT  = 'Explain conflict-free replicated data types (CRDTs) in two sentences.'

## 4. Run the head-to-head

Both calls fire concurrently. You get each response, token cost (from the stamped price snapshot), and wall-clock latency, side by side. **No verdict** — you judge the two responses yourself.

In [ ]:
from compare_core import compare_core, format_result

result = compare_core(PROMPT, [MODEL_A, MODEL_B])
print(format_result(result))

---

**Prices are a dated snapshot, not a live feed.** For the live, authoritative table see [llmprover.com/api-pricing/](https://llmprover.com/api-pricing/).

Want to compare 5-20 models at once, score which is best, see which layer broke, track drift over time, or run this inside your agent? That is [LLM Prover](https://llmprover.com). Not sure which models to use? See [llmprover.com/models/](https://llmprover.com/models/).